# Assignment 2: Neural Sequence Models

This notebook implements the **Bidirectional LSTM** part of Assignment 2 for the caption–question relevance task on RSVLM-QA, and provides scaffolding for adding a Transformer/BERT model later.

We reuse the same dataset, splits, and primary metric (F1-macro) as in Assignment 1.

## 1. Setup and Imports

Set random seeds, import libraries, and configure the device (CPU/GPU).

In [13]:
import math
import random
import time
from collections import Counter
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Using device: cpu



## 2. Load RSVLM-QA data and construct the classification dataset

We replicate the dataset from Assignment 1:
- **Positive pairs (label=1)**: correct caption–question–answer triplets from the same image
- **Negative pairs (label=0)**: caption from a different image paired with the question–answer
- **Text field**: `caption [SEP] question answer` (same format as A1)


In [14]:

# ---------------------------------------------------------------------------
# Reproduce the EXACT same dataset construction from Assignment 1
# ---------------------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 1) Load both Parquet files
df_captions = pd.read_parquet("RSVLM-QA-captions.parquet")
df_qa = pd.read_parquet("RSVLM-QA-questions.parquet")

print("Captions shape:", df_captions.shape, "  columns:", df_captions.columns.tolist())
print("QA shape:      ", df_qa.shape, "  columns:", df_qa.columns.tolist())

# 2) POSITIVE examples: merge captions with QA on shared 'id'
df_positive = df_qa.merge(df_captions[["id", "caption"]], on="id", how="inner")
df_positive["question_answer"] = df_positive["question"].astype(str) + " " + df_positive["answer"].astype(str)
df_positive["label"] = 1

# 3) NEGATIVE examples: pair each QA with a RANDOM distant caption
#    (A1 used tag-embedding semantic distance; random is a safe fallback
#     that still creates valid negatives without needing FastText.)
all_caption_ids = df_captions["id"].values
caption_lookup = df_captions.set_index("id")["caption"]

df_negative = df_qa.copy()
rng = np.random.RandomState(RANDOM_STATE)
neg_captions = []
for _, row in df_qa.iterrows():
    # pick a random caption that does NOT belong to the same image
    while True:
        rand_id = rng.choice(all_caption_ids)
        if rand_id != row["id"]:
            break
    neg_captions.append(caption_lookup[rand_id])

df_negative["caption"] = neg_captions
df_negative["question_answer"] = df_negative["question"].astype(str) + " " + df_negative["answer"].astype(str)
df_negative["label"] = 0

# 4) Combine, shuffle, build the text field matching A1's format
df_combined = pd.concat([df_positive, df_negative], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Ensure all text columns are strings (handles NaN)
df_combined["caption"] = df_combined["caption"].fillna("").astype(str)
df_combined["question_answer"] = df_combined["question_answer"].fillna("").astype(str)
df_combined["text"] = df_combined["caption"] + " [SEP] " + df_combined["question_answer"]

print(f"\nTotal dataset size: {len(df_combined)}")
print("Label distribution:\n", df_combined["label"].value_counts())
print("\nExample text:", df_combined["text"].iloc[0][:200])


Captions shape: (13820, 4)   columns: ['id', 'image', 'caption', 'tags']
QA shape:       (148558, 4)   columns: ['id', 'question_type', 'question', 'answer']

Total dataset size: 297116
Label distribution:
 label
0    148558
1    148558
Name: count, dtype: int64

Example text: The image primarily features a high-density residential complex with several large apartment buildings arranged in a geometric pattern, occupying much of the central and upper parts of the scene. Road



## 3. Train / Validation / Test split

We first reproduce Assignment 1's **80/20 stratified** train/test split (`random_state=42`), then carve a **10% validation** set from the training portion for early stopping.


In [15]:

# ---------------------------------------------------------------------------
# Replicate A1's 80/20 train/test split, then carve 10% validation from train
# ---------------------------------------------------------------------------
X_all = df_combined["text"].tolist()
y_all = df_combined["label"].tolist()

# Step 1: Exact same 80/20 split as Assignment 1
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
)

# Step 2: Split training set → 90% train / 10% validation (for early stopping)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f"Train size: {len(X_train)}")
print(f"Val size:   {len(X_val)}")
print(f"Test size:  {len(X_test)}")


Train size: 213922
Val size:   23770
Test size:  59424


## 4. Vocabulary and Tokenizer

We build a simple lowercase whitespace tokenizer and a frequency-based vocabulary with special tokens `<PAD>` and `<UNK>`.

In [16]:
def simple_tokenizer(text) -> List[str]:
    return str(text).lower().split()


def build_vocab(texts: List[str], min_freq: int = 2, max_size: int = 20000) -> Tuple[Dict[str, int], callable]:
    counter: Counter = Counter()
    for t in texts:
        counter.update(simple_tokenizer(t))

    vocab: Dict[str, int] = {"<PAD>": 0, "<UNK>": 1}

    for token, freq in counter.most_common():
        if freq < min_freq:
            continue
        if len(vocab) >= max_size:
            break
        if token not in vocab:
            vocab[token] = len(vocab)

    return vocab, simple_tokenizer


vocab, tokenizer = build_vocab(X_train, min_freq=2, max_size=20000)
pad_index = vocab["<PAD>"]
unk_index = vocab["<UNK>"]
print("Vocab size:", len(vocab))

Vocab size: 10237


## 5. Dataset and DataLoaders

We define a `Dataset` returning token indices, sequence lengths, and labels, and a `collate_fn` to pad batches to the same length.

In [17]:
class TextDataset(Dataset):
    def __init__(
        self,
        texts: List[str],
        labels: List[int],
        vocab: Dict[str, int],
        tokenizer,
        max_len: int = 128,
        pad_idx: int = 0,
    ) -> None:
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pad_idx = pad_idx

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        text = self.texts[idx]
        label = int(self.labels[idx])
        tokens = self.tokenizer(text)[: self.max_len]
        ids = [self.vocab.get(tok, self.vocab.get("<UNK>", 1)) for tok in tokens]
        length = len(ids)
        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long),
        }


def collate_batch(batch, pad_idx: int = 0):
    ids = [item["ids"] for item in batch]
    lengths = torch.tensor([item["length"].item() for item in batch], dtype=torch.long)
    labels = torch.tensor([item["label"].item() for item in batch], dtype=torch.long)

    padded = nn.utils.rnn.pad_sequence(
        ids, batch_first=True, padding_value=pad_idx
    )
    return padded, lengths, labels


train_ds = TextDataset(X_train, y_train, vocab, tokenizer, max_len=128, pad_idx=pad_index)
val_ds = TextDataset(X_val, y_val, vocab, tokenizer, max_len=128, pad_idx=pad_index)
test_ds = TextDataset(X_test, y_test, vocab, tokenizer, max_len=128, pad_idx=pad_index)

batch_size = 64

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    collate_fn=lambda b: collate_batch(b, pad_idx=pad_index),
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size * 2, shuffle=False,
    collate_fn=lambda b: collate_batch(b, pad_idx=pad_index),
)
test_loader = DataLoader(
    test_ds, batch_size=batch_size * 2, shuffle=False,
    collate_fn=lambda b: collate_batch(b, pad_idx=pad_index),
)

len(train_loader), len(val_loader), len(test_loader)

(3343, 186, 465)

## 6. BiLSTM Model

We implement the Bidirectional LSTM classifier with packed sequences and concatenated final forward/backward hidden states.

In [18]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 200,
        hidden_dim: int = 256,
        num_classes: int = 2,
        num_layers: int = 2,
        dropout: float = 0.3,
        padding_idx: int = 0,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, (hidden, _cell) = self.lstm(packed)
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        final_hidden = torch.cat([forward_hidden, backward_hidden], dim=1)
        output = self.dropout(final_hidden)
        logits = self.fc(output)
        return logits


model = LSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=200,  # to tune
    hidden_dim=256,     # to tune
    num_classes=2,
    num_layers=2,       # to tune (1/2/3)
    dropout=0.3,        # to tune (0.1/0.3/0.5)
    padding_idx=pad_index,
).to(device)
model

LSTMClassifier(
  (embedding): Embedding(10237, 200, padding_idx=0)
  (lstm): LSTM(200, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=2, bias=True)
)

## 7. Training Utilities (Early Stopping, Train/Eval Loops)

In [19]:
class EarlyStopping:
    def __init__(self, patience: int = 5, min_delta: float = 1e-3) -> None:
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.best_state = None
        self.early_stop = False

    def step(self, score: float, model: nn.Module) -> bool:
        if self.best_score is None:
            self.best_score = score
            self.best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            return False
        if score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        else:
            self.best_score = score
            self.best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            self.counter = 0
        return False


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_grad_norm: float = 1.0,
) -> float:
    model.train()
    total_loss = 0.0
    for inputs, lengths, labels in dataloader:
        inputs = inputs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(dataloader))


@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    all_labels: List[int] = []
    all_preds: List[int] = []
    for inputs, lengths, labels in dataloader:
        inputs = inputs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        logits = model(inputs, lengths)
        loss = criterion(logits, labels)
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
    avg_loss = total_loss / max(1, len(dataloader))
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1

## 8. Train the BiLSTM

We now train the model, monitor train/val loss and F1-macro, and apply early stopping based on validation F1-macro.

In [20]:
from tqdm.auto import tqdm

learning_rate = 3e-4  # to tune
num_epochs = 30
max_grad_norm = 1.0

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
early_stopper = EarlyStopping(patience=5, min_delta=1e-3)

history = {"train_loss": [], "val_loss": [], "val_f1": []}

epoch_bar = tqdm(range(num_epochs), desc="Training", unit="epoch")
for epoch in epoch_bar:
    # --- train ---
    model.train()
    running_loss = 0.0
    batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}", leave=False, unit="batch")
    for inputs, lengths, labels in batch_bar:
        inputs = inputs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        running_loss += loss.item()
        batch_bar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss = running_loss / max(1, len(train_loader))

    # --- validate ---
    val_loss, val_f1 = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    epoch_bar.set_postfix(
        train_loss=f"{train_loss:.4f}",
        val_loss=f"{val_loss:.4f}",
        val_f1=f"{val_f1:.4f}",
    )

    if early_stopper.step(val_f1, model):
        print(f"\nEarly stopping triggered at epoch {epoch+1}.")
        break

if early_stopper.best_state is not None:
    model.load_state_dict(early_stopper.best_state)
    print(f"Loaded best model (val_f1={early_stopper.best_score:.4f}).")

c:\Users\ABC\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Training:   0%|          | 0/30 [08:13<?, ?epoch/s]


KeyboardInterrupt: 

## 9. Training Curves (Optional Plot)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(epochs, history["val_f1"], label="Val F1-macro")
plt.xlabel("Epoch")
plt.ylabel("F1-macro")
plt.title("Validation F1-macro")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Final Test Evaluation

We evaluate the best BiLSTM model on the held-out test set and report F1-macro (primary metric).

In [ ]:
test_loss, test_f1 = evaluate(model, test_loader, criterion, device)
print(f"Test loss = {test_loss:.4f}")
print(f"Test F1-macro = {test_f1:.4f}")

## 11. Placeholder: Transformer / BERT Model (To Be Implemented)

In the next step of the assignment you will:
- Implement a Transformer encoder classifier **or** fine-tune a pre-trained model like BERT/DistilBERT.
- Reuse the same train/val/test splits and evaluation metric.
- Compare performance, learning curves, ablations, and computational cost against this BiLSTM and your Assignment 1 baselines.